In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv("HomeC.csv", low_memory=False)
df.head()

,time,use [kW],gen [kW],House overall [kW],Dishwasher [kW],Furnace 1 [kW],Furnace 2 [kW],Home office [kW],Fridge [kW],Wine cellar [kW],...,visibility,summary,apparentTemperature,pressure,windSpeed,cloudCover,windBearing,precipIntensity,dewPoint,precipProbability
0,1451624400,0.932833,0.003483,0.932833,0.000033,0.020700,0.061917,0.442633,0.124150,0.006983,...,10.0,Clear,29.26,1016.91,9.18,cloudCover,282,0.0,24.4,0.0
1,1451624401,0.934333,0.003467,0.934333,0.000000,0.020717,0.063817,0.444067,0.124000,0.006983,...,10.0,Clear,29.26,1016.91,9.18,cloudCover,282,0.0,24.4,0.0
2,1451624402,0.931817,0.003467,0.931817,0.000017,0.020700,0.062317,0.446067,0.123533,0.006983,...,10.0,Clear,29.26,1016.91,9.18,cloudCover,282,0.0,24.4,0.0
3,1451624403,1.022050,0.003483,1.022050,0.000017,0.106900,0.068517,0.446583,0.123133,0.006983,...,10.0,Clear,29.26,1016.91,9.18,cloudCover,282,0.0,24.4,0.0
4,1451624404,1.139400,0.003467,1.139400,0.000133,0.236933,0.063983,0.446533,0.122850,0.006850,...,10.0,Clear,29.26,1016.91,9.18,cloudCover,282,0.0,24.4,0.0


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503910 entries, 0 to 503909
Data columns (total 32 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   time                 503910 non-null  int64  
 1   use [kW]             503910 non-null  float64
 2   gen [kW]             503910 non-null  float64
 3   House overall [kW]   503910 non-null  float64
 4   Dishwasher [kW]      503910 non-null  float64
 5   Furnace 1 [kW]       503910 non-null  float64
 6   Furnace 2 [kW]       503910 non-null  float64
 7   Home office [kW]     503910 non-null  float64
 8   Fridge [kW]          503910 non-null  float64
 9   Wine cellar [kW]     503910 non-null  float64
 10  Garage door [kW]     503910 non-null  float64
 11  Kitchen 12 [kW]      503910 non-null  float64
 12  Kitchen 14 [kW]      503910 non-null  float64
 13  Kitchen 38 [kW]      503910 non-null  float64
 14  Barn [kW]            503910 non-null  float64
 15  Well [kW]        

In [14]:
df.isnull().sum()

time                   0
use [kW]               0
gen [kW]               0
House overall [kW]     0
Dishwasher [kW]        0
Furnace 1 [kW]         0
Furnace 2 [kW]         0
Home office [kW]       0
Fridge [kW]            0
Wine cellar [kW]       0
Garage door [kW]       0
Kitchen 12 [kW]        0
Kitchen 14 [kW]        0
Kitchen 38 [kW]        0
Barn [kW]              0
Well [kW]              0
Microwave [kW]         0
Living room [kW]       0
Solar [kW]             0
temperature            0
icon                   0
humidity               0
visibility             0
summary                0
apparentTemperature    0
pressure               0
windSpeed              0
cloudCover             0
windBearing            0
precipIntensity        0
dewPoint               0
precipProbability      0
dtype: int64

In [15]:
df.duplicated().sum()

np.int64(0)

In [16]:
# clean columns for clean text  [kw]
import re

df.columns = [re.sub(r'\[.*?\]','',col).replace(' ','_').replace('-','_').strip() for col in df.columns]

In [17]:
# Drop the duplicate column safely
df.drop(columns=['House_overall_'], errors='ignore', inplace=True)

In [18]:
df['cloudCover'] = pd.to_numeric(df['cloudCover'], errors = 'coerce')

In [19]:
df['time'] = pd.to_datetime(df['time'], unit='s')
df.set_index('time', inplace=True)

In [20]:
#  columns that contain any missing values data points in last of the dataset
missing_data = df.isnull().sum()
print(missing_data[missing_data>0])

cloudCover    58
dtype: int64


In [21]:
#  list of the appliances we want to incldue in our app 
appliance_cols = ['Dishwasher_', 'Furnace_1_', 'Furnace_2_', 'Home_office_', 'Fridge_']
df[appliance_cols].describe()

,Dishwasher_,Furnace_1_,Furnace_2_,Home_office_,Fridge_
count,503910.000000,503910.000000,503910.000000,503910.000000,503910.000000
mean,0.031368,0.099210,0.136779,0.081287,0.063556
std,0.190951,0.169059,0.178631,0.104466,0.076199
min,0.000000,0.000017,0.000067,0.000083,0.000067
25%,0.000000,0.020233,0.064400,0.040383,0.005083
50%,0.000017,0.020617,0.066633,0.042217,0.005433
75%,0.000233,0.068733,0.080633,0.068283,0.125417
max,1.401767,1.934083,0.794933,0.971750,0.851267


In [23]:
# map the thresholds
thresholds = {
    'Dishwasher_':0.05,
    'Furnace_1_':0.10,
    'Furnace_2_':0.10,
    'Home_office_':0.05,
    'Fridge_':0.05,
}

# run a loop to create the binary cols (0 or 1)
for appliance, thresh in thresholds.items():
    # this create a clear column name like "is_dishwasher_on"
    binary_col_name = f"is_{appliance.rstrip('_')}_on"
    df[binary_col_name] = (df[appliance] > thresh).astype(int)

# print a sample to verify the original vs the new binary column
print("binary column succesfully reated ! ")
df[['Dishwasher_', 'is_Dishwasher_on', 'Furnace_1_', 'is_Furnace_1_on']].head()

binary column succesfully reated ! 


,Dishwasher_,is_Dishwasher_on,Furnace_1_,is_Furnace_1_on
time,,,,
2016-01-01 05:00:00,0.000033,0,0.020700,0
2016-01-01 05:00:01,0.000000,0,0.020717,0
2016-01-01 05:00:02,0.000017,0,0.020700,0
2016-01-01 05:00:03,0.000017,0,0.106900,1
2016-01-01 05:00:04,0.000133,0,0.236933,1


In [24]:
# we need to predict a monthly bill based on daily appliance usage hours.
import pandas as pd
# start a clean dict. to hold our daily aggregation instructions
agg_rules = {
    'use_':'sum',     # total raw power baseline for a day
    'temperature':'mean',    # avg. daily temp.
    'humidity': 'mean',      # avg. daily humidity
    'cloudCover': 'mean'     # avg. daily cloudcover
}

# dymamically add our appliance binary cols to the rules (we can sum their active minutes)
binary_cols = [col for col in df.columns if col.startswith('is_')]
for col in binary_cols:
    agg_rules[col] = 'sum'

# resample the 1-minute data into daily data columns
daily_df = df.resample('D').agg(agg_rules)

# convert the summed active minutes into operational hours (minutes / 60)
for col in binary_cols:
    hour_col_name = col.replace('is_', '').replace('_on','_hours')   #e.g., 'is_Dishwasher_on' -> 'Dishwasher_hours
    daily_df[hour_col_name] = daily_df[col] / 60
    # drop the temporary sum column to keep it clean 
    daily_df.drop(columns=[col], inplace=True)

# take a look at your brand-new , engineered daily dataset
daily_df.head()

,use_,temperature,humidity,cloudCover,Dishwasher_hours,Furnace_1_hours,Furnace_2_hours,Home_office_hours,Fridge_hours
time,,,,,,,,,
2016-01-01,70947.811883,27.548216,0.628002,0.201543,61.083333,564.583333,436.150000,392.383333,441.083333
2016-01-02,69218.821350,41.630617,0.581337,0.231187,70.083333,515.816667,867.250000,339.083333,545.416667
2016-01-03,49438.172884,60.047838,0.625934,0.252554,64.316667,109.533333,41.033333,351.383333,628.233333
2016-01-04,95283.612300,72.961776,0.687193,0.187292,62.700000,126.900000,96.916667,560.316667,795.483333
2016-01-05,73720.802267,61.443197,0.737395,0.198290,71.533333,99.033333,52.950000,639.016667,727.483333


In [25]:
# Creating our Target Variable (The Financial Cost)

# We group by date and find the mean of the binary entries, then multiply by 24 hours!
daily_df_fixed = pd.DataFrame()

# Re-calculate environmental averages safely
daily_df_fixed['temperature'] = df['temperature'].resample('D').mean()
daily_df_fixed['humidity'] = df['humidity'].resample('D').mean()
daily_df_fixed['cloudCover'] = df['cloudCover'].resample('D').mean()

# 2. FIXED APPLIANCE MATH: Calculate actual hours out of a 24-hour day
binary_cols = [col for col in df.columns if col.startswith('is_')]
for col in binary_cols:
    hour_col_name = col.replace('is_', '').replace('_on', '_hours')
    # .mean() gives the fraction of time it was ON (e.g., 0.10 of the day). Multiplying by 24 gives true hours!
    daily_df_fixed[hour_col_name] = df[col].resample('D').mean() * 24

# 3. FIXED COST MATH: Calculate realistic daily cost based on total consumption fraction
ENERGY_RATE = 8.00
# In this dataset, the 'use' is in kW. Average kW over the day * 24 hours = total unit consumption (kWh)
daily_df_fixed['Daily_Cost'] = df['use_'].resample('D').mean() * 24 * ENERGY_RATE

# 4. Overwrite our old dataframe with the clean version
daily_df = daily_df_fixed

# 5. Let's inspect the beautifully corrected dataset
daily_df.head()

,temperature,humidity,cloudCover,Dishwasher_hours,Furnace_1_hours,Furnace_2_hours,Home_office_hours,Fridge_hours,Daily_Cost
time,,,,,,,,,
2016-01-01,27.548216,0.628002,0.201543,1.285965,11.885965,9.182105,8.260702,9.285965,199.151753
2016-01-02,41.630617,0.581337,0.231187,1.168056,8.596944,14.454167,5.651389,9.090278,153.819603
2016-01-03,60.047838,0.625934,0.252554,1.071944,1.825556,0.683889,5.856389,10.470556,109.862606
2016-01-04,72.961776,0.687193,0.187292,1.045000,2.115000,1.615278,9.338611,13.258056,211.741361
2016-01-05,61.443197,0.737395,0.198290,1.192222,1.650556,0.882500,10.650278,12.124722,163.824005


In [26]:
# One-Hot Encoding the Weather Data

#  take the most comman weather condition for each day from our original dataframs
daily_weather = df['summary'].resample('D').apply(lambda x:x.mode()[0] if not x.empty else 'clear')

# use one-hot encoding to turn those text labels into columns of 0s and 1s
weather_dummies = pd.get_dummies(daily_weather, prefix='weather')

# join this new weather col directly into our clean daily_df
daily_df = daily_df.join(weather_dummies)

# let's see your complete , final output
daily_df.head()

,temperature,humidity,cloudCover,Dishwasher_hours,Furnace_1_hours,Furnace_2_hours,Home_office_hours,Fridge_hours,Daily_Cost,weather_Clear
time,,,,,,,,,,
2016-01-01,27.548216,0.628002,0.201543,1.285965,11.885965,9.182105,8.260702,9.285965,199.151753,True
2016-01-02,41.630617,0.581337,0.231187,1.168056,8.596944,14.454167,5.651389,9.090278,153.819603,True
2016-01-03,60.047838,0.625934,0.252554,1.071944,1.825556,0.683889,5.856389,10.470556,109.862606,True
2016-01-04,72.961776,0.687193,0.187292,1.045000,2.115000,1.615278,9.338611,13.258056,211.741361,True
2016-01-05,61.443197,0.737395,0.198290,1.192222,1.650556,0.882500,10.650278,12.124722,163.824005,True


In [27]:
# save the dataset to a clean csv file
daily_df.to_csv('cleaned_daily_energy_data.csv', index=True)
print("dataset successfully saved !")

dataset successfully saved !


In [29]:
# lets try to reload the main dataset file to check its full size
try:
    # adjust this filename to match your origianl raw smart home dataset file if named differently
    full_df = pd.read_csv('homeC.csv')
    print(f"success ! full dataset loaded.")
    print(f"total raw records found: {full_df.shape[0]:,}rows")
except Exception as e:
    print(f"could not find the raw file. error details: {e}")

C:\Users\payal\AppData\Local\Temp\ipykernel_21424\480979044.py:4: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  full_df = pd.read_csv('homeC.csv')


success ! full dataset loaded.
total raw records found: 503,910rows


In [30]:
import pandas as pd
import numpy as np

# 1. Clean up index and column names on the full dataset
if 'time' in full_df.columns:
    full_df['time'] = pd.to_datetime(full_df['time'], unit='s', errors='coerce')
    full_df.set_index('time', inplace=True)
full_df.columns = full_df.columns.str.strip()

# Helper function to find a column name even with weird formatting/underscores
def find_col(name):
    for c in full_df.columns:
        if name.lower() in c.lower():
            return c
    return None

# 2. Automatically find your exact column keys from the CSV
col_use = find_col('use')
col_temp = find_col('temperature')
col_hum = find_col('humidity')
col_cloud = find_col('cloudCover')

col_dish = find_col('Dishwasher')
col_furn1 = find_col('Furnace_1') or find_col('Furnace 1') or find_col('Furnace_1_')
col_furn2 = find_col('Furnace_2') or find_col('Furnace 2') or find_col('Furnace_2_')
col_fridge = find_col('Fridge')
col_office = find_col('Home_office') or find_col('Home office')

# 3. Force them to be numeric data types and fill missing points
all_found_cols = [col_use, col_temp, col_hum, col_cloud, col_dish, col_furn1, col_furn2, col_fridge, col_office]
for col in all_found_cols:
    if col:
        full_df[col] = pd.to_numeric(full_df[col], errors='coerce').ffill()

# 4. Create the binary ON/OFF columns using those detected names safely
full_df['is_Dishwasher_on']  = (full_df[col_dish] > 0.05).astype(int) if col_dish else 0
full_df['is_Furnace_1_on']   = (full_df[col_furn1] > 0.10).astype(int) if col_furn1 else 0
full_df['is_Furnace_2_on']   = (full_df[col_furn2] > 0.10).astype(int) if col_furn2 else 0
full_df['is_Fridge_on']      = (full_df[col_fridge] > 0.05).astype(int) if col_fridge else 0
full_df['is_Home_office_on'] = (full_df[col_office] > 0.05).astype(int) if col_office else 0

# 5. Squeeze into Daily data intervals ('D') using our fraction math
daily_df = pd.DataFrame()
if col_temp:  daily_df['temperature'] = full_df[col_temp].resample('D').mean()
if col_hum:   daily_df['humidity'] = full_df[col_hum].resample('D').mean()
if col_cloud: daily_df['cloudCover'] = full_df[col_cloud].resample('D').mean()

# Calculate true daily operational hours explicitly
daily_df['Dishwasher_hours']  = full_df['is_Dishwasher_on'].resample('D').mean() * 24
daily_df['Furnace_1_hours']   = full_df['is_Furnace_1_on'].resample('D').mean() * 24
daily_df['Furnace_2_hours']   = full_df['is_Furnace_2_on'].resample('D').mean() * 24
daily_df['Fridge_hours']      = full_df['is_Fridge_on'].resample('D').mean() * 24
daily_df['Home_office_hours'] = full_df['is_Home_office_on'].resample('D').mean() * 24

# Calculate true Daily_Cost based on complete consumption
ENERGY_RATE = 8.00
if col_use:
    daily_df['Daily_Cost'] = full_df[col_use].resample('D').mean() * 24 * ENERGY_RATE

# 6. One-Hot Encode weather categories safely
col_summary = find_col('summary')
if col_summary and col_summary in full_df.columns:
    daily_weather = full_df[col_summary].resample('D').apply(lambda x: x.mode()[0] if not x.empty else 'Clear')
    weather_dummies = pd.get_dummies(daily_weather, prefix='weather')
    daily_df = daily_df.join(weather_dummies)

print("🚀 BOOM! Setup complete with no key errors. Your full dataset is ready!")
daily_df.head()

🚀 BOOM! Setup complete with no key errors. Your full dataset is ready!


,temperature,humidity,cloudCover,Dishwasher_hours,Furnace_1_hours,Furnace_2_hours,Fridge_hours,Home_office_hours,Daily_Cost,weather_Clear
time,,,,,,,,,,
2016-01-01,27.548216,0.628002,0.201543,1.285965,11.885965,9.182105,9.285965,8.260702,199.151753,True
2016-01-02,41.630617,0.581337,0.231187,1.168056,8.596944,14.454167,9.090278,5.651389,153.819603,True
2016-01-03,60.047838,0.625934,0.252554,1.071944,1.825556,0.683889,10.470556,5.856389,109.862606,True
2016-01-04,72.961776,0.687193,0.187292,1.045000,2.115000,1.615278,13.258056,9.338611,211.741361,True
2016-01-05,61.443197,0.737395,0.198290,1.192222,1.650556,0.882500,12.124722,10.650278,163.824005,True


In [31]:
# Re-running the Training Engine

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# 1. Separate features (x) and target (y) from our fixed, full dataset
x = daily_df.drop(columns=['Daily_Cost'], errors='ignore')
y = daily_df['Daily_Cost']

# 2. Split the full dataset into 80% training and 20% testing
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# 3. Initialize and train the model on the full data
model = LinearRegression()
model.fit(x_train, y_train)

print("🎉 MODEL TRAINED SUCCESSFULLY ON FULL DATASET!")
print(f"Total historical days available for training: {x_train.shape[0]} days\n")

# 4. Display the true, fixed intercept
print(f"Model Intercept (Real Base Daily Standby Cost): ₹{model.intercept_:.2f}")

print("\n--- 🧠 REAL APPLIANCE WEIGHTS (COEFFICIENTS) ---")
# 5. Loop through and print the real mathematical weights
for feature, weight in zip(x.columns, model.coef_):
    print(f"{feature}: {weight:.4f}")

🎉 MODEL TRAINED SUCCESSFULLY ON FULL DATASET!
Total historical days available for training: 5 days

Model Intercept (Real Base Daily Standby Cost): ₹-330.37

--- 🧠 REAL APPLIANCE WEIGHTS (COEFFICIENTS) ---
temperature: -2.3530
humidity: -9.1285
cloudCover: 3.0655
Dishwasher_hours: 6.3975
Furnace_1_hours: 18.3283
Furnace_2_hours: -6.9161
Fridge_hours: 56.1961
Home_office_hours: -6.4077
weather_Clear: 0.0000


In [32]:
# Updated diagnostic cell ignoring text columns safely
print("1. Raw index type:", full_df.index.dtype)
print("2. First 3 dates in full_df:\n", full_df.index[:3])
print("3. Last 3 dates in full_df:\n", full_df.index[-3:])

# FIXED: Added numeric_only=True to prevent text summary columns from breaking the math
print("\n4. Number of unique days generated after resampling:", len(full_df.resample('D').mean(numeric_only=True)))

1. Raw index type: datetime64[ns]
2. First 3 dates in full_df:
 DatetimeIndex(['2016-01-01 05:00:00', '2016-01-01 05:00:01',
               '2016-01-01 05:00:02'],
              dtype='datetime64[ns]', name='time', freq=None)
3. Last 3 dates in full_df:
 DatetimeIndex(['2016-01-07 00:58:27', '2016-01-07 00:58:28',
               '2016-01-07 00:58:29'],
              dtype='datetime64[ns]', name='time', freq=None)

4. Number of unique days generated after resampling: 7


In [33]:
# creating  a safe prediction function that caps values at 0 or a base standby fee
# Model Post-Processing Constraint.

def predict_daily_bill(slider_features):
    # prediction using out trained model 
    raw_prediction = model.predict([slider_features])[0]

    # if a maths return negavite value ---
    # default to a minimum realistic standby connection cost (e.g rs.50.00)
    final_bill = max(50.00, raw_prediction)
    return final_bill

# test the predictor with a dummy rows
sample_user_day = x_test.iloc[0]
predicted_cost = predict_daily_bill(sample_user_day)

print(f"raw model output ₹{model.predict([sample_user_day])[0]:.2f}")
print(f"Streamlit App Safe Display Output: ₹{predicted_cost:.2f}")
    

raw model output ₹231.17
Streamlit App Safe Display Output: ₹231.17


C:\Users\payal\OneDrive\Attachments\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
C:\Users\payal\OneDrive\Attachments\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [36]:
# Exporting and Saving Your Engine

import joblib

# save the trained model object to a binary file
joblib.dump(model, 'energy_regression_model.pkl')

# save the exact list of features names so your app knows the sliders order
feature_names = list(x.columns)
joblib.dump(feature_names, 'model_features.pkl')

print("success !! ")

success !! 
